Trial 2: Trying to pull data on more than 38 players

In [1]:
#we want to clean up the names. i.e. "Stephen Curry g" -> "stephen curry" 
#we also want to clean up the salaary. i.e. "$52,000,000" -> "52000000"
import pandas as pd
import re
import unicodedata
import time

def clean_name(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    name = name.lower()
    name = re.sub(r"[^a-z\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

def clean_salary(s):
    s = str(s)
    s = re.sub(r"[\$,]", "", s)
    return pd.to_numeric(s, errors="coerce")


# Basketball Reference uses 2026 for the 2025-26 season
season = 2026

per_game_url = f"https://www.basketball-reference.com/leagues/NBA_{season}_per_game.html"
advanced_url = f"https://www.basketball-reference.com/leagues/NBA_{season}_advanced.html"

# Pull tables
per_game = pd.read_html(per_game_url)[0]
advanced = pd.read_html(advanced_url)[0]

# Remove repeated header rows
per_game = per_game[per_game["Rk"] != "Rk"]
advanced = advanced[advanced["Rk"] != "Rk"]

# Keep the per-game features we want
per_game = per_game[[
    "Player",
    #"Age",
    "G",
    "PTS",
    "AST",
    "TRB",
    "BLK"
]]

per_game = per_game.rename(columns={
    "PTS": "PPG",
    "TRB": "REB",
    "BLK": "Blocks",
    "G": "Games_Played"
})

# Keep the advanced features we want
advanced = advanced[[
    "Player",
    "TS%",
    "WS",
    "USG%",
    "VORP"
]]

# Merge per-game and advanced stats
stats = per_game.merge(
    advanced,
    on="Player",
    how="inner"
)

# Clean names for merging with salary data
stats["name_clean"] = stats["Player"].apply(clean_name)

# Convert numeric columns
numeric_cols = [
    #"Age",
    "Games_Played",
    "PPG",
    "AST",
    "REB",
    "Blocks",
    "TS%",
    "WS",
    "USG%",
    "VORP"
]

for col in numeric_cols:
    stats[col] = pd.to_numeric(stats[col], errors="coerce")

# Drop duplicate players
stats = stats.drop_duplicates(subset=["name_clean"])

print(stats.head())
print(stats.shape)

                    Player  Games_Played   PPG  AST  REB  Blocks    TS%    WS  \
0              Luka Dončić          64.0  33.5  8.3  7.7     0.5  0.616   9.5   
1  Shai Gilgeous-Alexander          68.0  31.1  6.6  4.3     0.8  0.665  15.2   
2          Anthony Edwards          61.0  28.8  3.7  5.0     0.8  0.617   6.5   
3             Jaylen Brown          71.0  28.7  5.1  6.9     0.4  0.573   6.9   
4             Tyrese Maxey          70.0  28.3  6.6  4.1     0.8  0.588   8.7   

   USG%  VORP              name_clean  
0  38.1   6.6             luka doncic  
1  33.4   7.8  shai gilgeousalexander  
2  31.5   3.5         anthony edwards  
3  36.2   3.3            jaylen brown  
4  29.4   4.9            tyrese maxey  
(583, 11)


In [2]:
teams = [
    "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", "DET", "GSW",
    "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK",
    "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
]

salary_dfs = []

for team in teams:
    url = f"https://www.basketball-reference.com/contracts/{team}.html"

    try:
        table = pd.read_html(url)[0]

        # Flatten MultiIndex columns correctly
        new_cols = []
        for col in table.columns:
            if isinstance(col, tuple):
                level0, level1 = col

                if "Unnamed" not in str(level1):
                    new_cols.append(level1)
                else:
                    new_cols.append(level0)
            else:
                new_cols.append(col)

        table.columns = new_cols

        # Keep only what we need
        table = table[["Player", "2025-26"]].copy()
        table["Team"] = team

        salary_dfs.append(table)
        print(f"Pulled {team}")

        time.sleep(2)

    except Exception as e:
        print(f"Could not pull {team}: {e}")



Pulled ATL
Pulled BOS
Pulled BRK
Pulled CHO
Pulled CHI
Pulled CLE
Pulled DAL
Pulled DEN
Pulled DET
Pulled GSW
Pulled HOU
Pulled IND
Pulled LAC
Pulled LAL
Pulled MEM
Pulled MIA
Pulled MIL
Pulled MIN
Pulled NOP
Pulled NYK
Pulled OKC
Pulled ORL
Pulled PHI
Pulled PHO
Pulled POR
Pulled SAC
Pulled SAS
Pulled TOR
Pulled UTA
Pulled WAS


In [3]:
salary = pd.concat(salary_dfs, ignore_index=True)

# Remove repeated header rows
salary = salary[salary["Player"] != "Player"]

# Rename 2025-26 salary column
salary = salary.rename(columns={
    "2025-26": "Salary"
})

# Clean names and salaries
salary["name_clean"] = salary["Player"].apply(clean_name)
salary["Salary"] = salary["Salary"].apply(clean_salary)
salary["Salary_Millions"] = salary["Salary"] / 1_000_000

# Drop missing salaries(not the best practice)
salary = salary.dropna(subset=["Salary_Millions"])

# Remove duplicate players if needed
salary = salary.drop_duplicates(subset=["name_clean"])

# Keep final columns
salary = salary[["Player", "Team", "name_clean", "Salary", "Salary_Millions"]]

print(salary.head())
print(salary.shape)

                     Player Team               name_clean      Salary  \
0               CJ McCollum  ATL              cj mccollum  30666666.0   
1             Jalen Johnson  ATL            jalen johnson  30000000.0   
2          Jonathan Kuminga  ATL         jonathan kuminga  23799569.0   
3  Nickeil Alexander-Walker  ATL  nickeil alexanderwalker  15161800.0   
4            Onyeka Okongwu  ATL           onyeka okongwu  15000000.0   

   Salary_Millions  
0        30.666666  
1        30.000000  
2        23.799569  
3        15.161800  
4        15.000000  
(491, 5)


In [4]:
#Now we want to get matches from our stats data frame and our salary data pull
matches = set(stats["name_clean"]) & set(salary["name_clean"])

print("Stats players:", stats.shape[0])
print("Salary players:", salary.shape[0])
print("Matching players:", len(matches))
print(list(matches)[:20])

Stats players: 583
Salary players: 491
Matching players: 472
['ja morant', 'anthony davis', 'clint capela', 'luka garza', 'devin vassell', 'ariel hukporti', 'james wiseman', 'khaman maluach', 'tony bradley', 'josh minott', 'jared mccain', 'garrett temple', 'dominick barlow', 'tobias harris', 'christian koloko', 'bryce mcgowens', 'vit krejci', 'andre jackson jr', 'brandon miller', 'zeke nnaji']


In [5]:
#merging our stats data frame and our salary dataframe
df = stats.merge(
    salary,
    on="name_clean",
    how="inner",
    suffixes=("", "_salary")
)

df = df[[
    "Player",
    "PPG",
    "AST",
    "REB",
    "TS%",
    "WS",
    "USG%",
    "Games_Played",
    #"Age",
    "Blocks",
    "VORP",
    "Salary_Millions"
]]

print(df.head())
print(df.shape)

                    Player   PPG  AST  REB    TS%    WS  USG%  Games_Played  \
0              Luka Dončić  33.5  8.3  7.7  0.616   9.5  38.1          64.0   
1  Shai Gilgeous-Alexander  31.1  6.6  4.3  0.665  15.2  33.4          68.0   
2          Anthony Edwards  28.8  3.7  5.0  0.617   6.5  31.5          61.0   
3             Jaylen Brown  28.7  5.1  6.9  0.573   6.9  36.2          71.0   
4             Tyrese Maxey  28.3  6.6  4.1  0.588   8.7  29.4          70.0   

   Blocks  VORP  Salary_Millions  
0     0.5   6.6        45.999660  
1     0.8   7.8        38.333050  
2     0.8   3.5        45.550512  
3     0.4   3.3        53.142264  
4     0.8   4.9        37.958760  
(472, 11)


In [6]:
#setting up our fetures and labels for our linear regression(In this case we want a linear regression because we aren't predicting a yes or no question)
features = [
    "PPG",
    "AST",
    "REB",
    "TS%",
    "WS",
    "USG%",
    "Games_Played",
    #"Age",
    "Blocks",
    "VORP"
]

label = "Salary_Millions"

X = df[features]
y = df[label]

#True shooting and Usage come in as strings so convert them into numbers
for col in features + [label]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=features + [label])

In [7]:
#Now do the test train split(80% training and 20% test data)
from sklearn.model_selection import train_test_split
#Do the linear regression after
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

MAE: 5.9817704826058895
RMSE: 8.320599488225628
R^2: 0.6927448023003591


In [8]:
all_pred = model.predict(X)

results_all = X.copy()
results_all["Player"] = df.loc[X.index, "Player"]
results_all["Actual_Salary"] = y
results_all["Predicted_Salary"] = all_pred
results_all["Salary_Difference"] = results_all["Predicted_Salary"] - results_all["Actual_Salary"]

results_all = results_all[[
    "Player",
    "Actual_Salary",
    "Predicted_Salary",
    "Salary_Difference"
]]

print(results_all.sort_values("Salary_Difference", ascending=False).to_string())

                       Player  Actual_Salary  Predicted_Salary  Salary_Difference
16          Victor Wembanyama      13.376880         38.922628          25.545748
22             Keyonte George       4.278960         28.969864          24.690904
17                Deni Avdija      14.375000         34.841834          20.466834
48                Jalen Duren       6.483144         26.855352          20.372208
86            Dariq Whitehead       3.262560         22.135478          18.872918
67           Kevin Porter Jr.       5.134000         22.876430          17.742430
97          Russell Westbrook       2.296274         19.609855          17.313581
35             Shaedon Sharpe       8.399983         25.075815          16.675832
69               Ryan Rollins       4.000000         20.446932          16.446932
64                 Saddiq Bey       6.118644         22.072991          15.954347
23              Austin Reaves      13.937574         29.793693          15.856119
192          Luc